# Fine-tuning



## Play with an exiting pre-trained model (generate a text)
1) install the necessary dependencies
2) download a pre-trained T5 model,
3) run a simple text-to-text prediction.

In [ ]:
!pip install transformers torch sentencepiece datasets transformers[torch]

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")



In [4]:
def generate_text(input_text, max_length=50):

    input_ids = tokenizer(input_text, return_tensors="pt").input_ids

    ### PRINT OUT input_ids

    output_ids = model.generate(input_ids, max_length=max_length)

    ### PRINT OUT output_ids

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


example_input = "translate English to French: How are you?"
output_text = generate_text(example_input)

print("Input:", example_input)
print("Output:", output_text)

Input: translate English to French: What is your name?
Output: Quel est votre nom?


4) Try at least 5 other example inputs
- Example 1: "How are you?"
- Example 2: "What is your name?"
- Example 3: "Where is the nearest restaurant?"
- Example 4: "I love learning new things."
- Example 5: "This is a beautiful day."

## Fine-tuning of an existing model

Digging a little bit: T5 and the Prefix + Input Structure

T5 (Text-to-Text Transfer Transformer) is explicitly trained to follow a **prefix + input** format, guiding it to perform the correct NLP task.

**Why Prefixes Matter?** T5 was trained using structured prompts like:
- `translate English to French: How are you?` → `Comment allez-vous?`
- `summarize: The Eiffel Tower is in Paris.` → `Eiffel Tower is in Paris.`
- `question: Who discovered gravity? context: Isaac Newton discovered gravity.` → `Isaac Newton`
- `sentiment: I love this movie!` → `positive`

**Without a Prefix?**

❌ `How are you?` → (Unpredictable output)  
✅ `translate English to French: How are you?` → `Comment allez-vous?`

**Custom Prefixes** Fine-tune T5 with your own prefixes:
- `explain: What is...`
- `medical diagnosis: Patient has high fever...`

In [7]:
import torch
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

csv_filename = "explain_dataset.csv"

df = pd.read_csv(csv_filename)
dataset = Dataset.from_pandas(df)

def preprocess_function(examples):
    inputs = examples["Input"]
    targets = examples["Response"]

    model_inputs = tokenizer(inputs, max_length=64, truncation=True, padding="max_length")

    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length").input_ids

    model_inputs["labels"] = labels

    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

dataset_split = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

training_args = TrainingArguments(
    output_dir="./t5-fine-tuned",
    # evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_strategy="epoch",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./t5-custom-response")
tokenizer.save_pretrained("./t5-custom-response")

print("Fine-tuning complete! Model saved to ./t5-custom-response")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.243255


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete! Model saved to ./t5-custom-response


In [10]:
original_model = T5ForConditionalGeneration.from_pretrained("t5-small")
fine_tuned_model = T5ForConditionalGeneration.from_pretrained("./t5-custom-response")

def generate_response(model, input_text):
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_length=64)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_question = "explain: What is deep learning?"

original_output = generate_response(original_model, test_question)
fine_tuned_output = generate_response(fine_tuned_model, test_question)

print("Original Model Output:")
print(original_output)
print("\n\n\nFine-Tuned Model Output:")
print(fine_tuned_output)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Original Model Output:
Warum ist deep learning?



Fine-Tuned Model Output:
Deep learning is a subset of AI that enables systems to learn from data and improve without explicit programming.
